In [1]:
import os

In [3]:
os.chdir("../")

In [15]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender'

In [37]:
from dataclasses import dataclass
from pathlib import Path

In [38]:
# entity/config_entity.py
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    metrics_file: Path
    sample_size: int      
    n_recommendations: int 

In [39]:
from Book_recommender.constant import *
from Book_recommender.utils.common import read_yaml, create_directories

In [40]:
# configuration
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # same upto here for most pipeline

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(  # ← Capital M and E and C
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            metrics_file = config.metrics_file,
            sample_size = config.sample_size,     
            n_recommendations =  config.n_recommendations
        )

        return model_evaluation_config

In [41]:
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import joblib  # add this
from Book_recommender.logging.log import logger

In [ ]:
# Compponents
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def load_artifacts(self):
        self.knn = joblib.load("artifacts/model_trainer/knn_model.pkl")       
        self.tfidf_matrix = joblib.load("artifacts/data_transformation/tfidf_matrix.pkl")  

        self.data = pd.read_csv(self.config.data_path)
        logger.info("Artifacts loaded for evaluation")

    def precision_at_k(self, book_idx, k):
        distances, indices = self.knn.kneighbors(
            self.tfidf_matrix[book_idx],
            n_neighbors=k + 1
        )
        rec_indices = indices.flatten()[1:]

        seed_genre = self.data.iloc[book_idx]["genre_clean"]
        rec_genres = self.data.iloc[rec_indices]["genre_clean"]

        # stronger check — covers NaN, None, float, empty string
        if not isinstance(seed_genre, str) or seed_genre.strip() == "":
            return None

        matches = rec_genres.str.contains(seed_genre, case=False, na=False, regex = False).sum()

        return matches / k

    def diversity_score(self, book_idx, k):
        """
        Measures how different recommendations are from each other
        1.0 = all different, 0.0 = all identical
        """
        distances, indices = self.knn.kneighbors(
            self.tfidf_matrix[book_idx],
            n_neighbors=k + 1
        )
        rec_indices = indices.flatten()[1:]
        rec_vectors = self.tfidf_matrix[rec_indices]

        sim_matrix = cosine_similarity(rec_vectors)
        # average similarity excluding diagonal
        np.fill_diagonal(sim_matrix, 0)
        avg_sim = sim_matrix.sum() / (k * (k - 1))

        return 1 - avg_sim  # higher = more diverse

    def coverage(self):
        """
        What % of books appear in at least one recommendation list
        """
        sample = self.data.sample(
            min(self.config.sample_size, len(self.data)),
            random_state=42
        )
        recommended = set()

        for idx in sample.index:
            _, indices = self.knn.kneighbors(
                self.tfidf_matrix[idx],
                n_neighbors=self.config.n_recommendations + 1
            )
            recommended.update(indices.flatten()[1:].tolist())

        return len(recommended) / len(self.data)

    def run_evaluation(self):
        self.load_artifacts()

        sample = self.data.sample(
            min(self.config.sample_size, len(self.data)),
            random_state=42
        )

        precisions = []
        diversities = []

        for idx in sample.index:
            p = self.precision_at_k(idx, self.config.n_recommendations)
            d = self.diversity_score(idx, self.config.n_recommendations)

            if p is not None:
                precisions.append(p)
            diversities.append(d)

        metrics = {
            "precision_at_k":   round(np.mean(precisions), 4),
            "diversity_score":  round(np.mean(diversities), 4),
            "catalog_coverage": round(self.coverage(), 4),
            "k":                self.config.n_recommendations,
            "sample_size":      len(sample)
        }

        # save metrics
        with open(self.config.metrics_file, "w") as f:
            json.dump(metrics, f, indent=4)

        logger.info(f"Evaluation complete: {metrics}")
        return metrics


In [43]:
# Pipeline
try:
    config = configurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    
    if not os.path.exists(model_evaluation_config.metrics_file):
        logger.info("No metrics found — running evaluation...")
        model_evaluation = ModelEvaluation(config=model_evaluation_config)
        metrics = model_evaluation.run_evaluation()
        print(metrics)
    else:
        logger.info("Metrics already exist — skipping evaluation!")

except Exception as e:
    raise e

[2026-06-02 20:47:15,190: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-06-02 20:47:15,193: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-06-02 20:47:15,197: INFO: common: created directory at artifacts]
[2026-06-02 20:47:15,199: INFO: common: created directory at artifacts/model_evaluation]
[2026-06-02 20:47:15,201: INFO: 760023260: No metrics found — running evaluation...]
[2026-06-02 20:47:15,275: INFO: 3610473802: Artifacts loaded for evaluation]
[2026-06-02 20:47:18,604: INFO: 3610473802: Evaluation complete: {'precision_at_k': np.float64(0.4289), 'diversity_score': np.float64(0.6147), 'catalog_coverage': 0.0852, 'k': 5, 'sample_size': 200}]
{'precision_at_k': np.float64(0.4289), 'diversity_score': np.float64(0.6147), 'catalog_coverage': 0.0852, 'k': 5, 'sample_size': 200}


In [45]:
# run this to verify
import pandas as pd
df = pd.read_csv("artifacts/data_preprocessing/processed_data.csv")
print(df.columns.tolist())

['title', 'authors', 'average_rating', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'description', 'genre_clean', 'combined_features']
